Intentando crear notebook para procesar los datos del recording de unity y entrenarlos.

Utilizando como base:
- Tutorial de Pablo en la carpeta de tutoriales
- Ejercicios https://github.com/dair-ai/pytorch_notebooks?tab=readme-ov-file
- Puede ser util: https://medium.com/@flexianadevgroup/setting-up-a-python-environment-for-ai-development-a-comprehensive-guide-022602e337f4 
- Crear environment: https://docs.conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html

> PASOS A DAR

He instalado miniconda: https://www.anaconda.com/download/success

En la carpeta de entrenamiento esta el .yml del environment del tutorial. 

Para crearlo: conda env create -f dl2024_gpu.yml

Para activarlo: elegir kernel 

In [1]:
## Standard libraries
import os
import math
import numpy as np
import time
import datetime

## Imports for plotting
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline
#from IPython.display import set_matplotlib_formats
#set_matplotlib_formats('svg', 'pdf') # For export
from matplotlib.colors import to_rgba
import seaborn as sns
sns.set_theme()

# Asegurar fuente disponible y permitir fallback
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"]
mpl.rcParams["text.usetex"] = False
# Usar STIX sans para mathtext, evita dependencias en DejaVu
mpl.rcParams["mathtext.fontset"] = "stixsans"
mpl.rcParams["svg.fonttype"] = "none"

## Progress bar
from tqdm.notebook import tqdm

In [2]:
import torch
print("Using torch", torch.__version__)

#Para poder reproducir nuestro código con los mismos números random siempre
torch.manual_seed(42) # Setting the seed

if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    #torch.cuda.manual_seed_all(42)     # all GPUs

    # Additionally, some operations on a GPU are implemented stochastic for efficiency
    # We want to ensure that all operations are deterministic on GPU (if used) for reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

elif torch.mps.is_available():
    torch.mps.manual_seed(42)

Using torch 2.5.0


*TENSORES*

Un tensor es prácticamente una matriz d N dimensiones, pero optimizado para machine learning con GPUs. Entiendo que en nuestro caso necesitaríamos un tensor de (n puntos de la tela * n d caracteristicas que queramos guardar * n de frames).

- Reservan su propia memoria (reutilizan valores...) Interesante para mencionarlo en el tfg, tmbn para plantear si queremos redondear nuestros valores para ahorrar.
- Lo txulo d los tensores es q calculan automaticamente los gradientes (output values) de las funcones para el backpropagation.
- Los tensores funcionan en gpu, pero se pueden pasar a cpu si queremos utilizarlos como numpy arrays. (Ns si necesitaremos hacer esto pero vaya)
- Operaciones comunes: Sumas, reordenar en distintas dimensiones, mult d matrices...

En el tutorial habla d "computation graph" q es básicamente hacer calculos a mano cn el tensor para hacer el backpropagation de este grafo y sacar los gradientes "a mano". No entiendo como trasladar esto a nuestro caso en el que los cálculos ya los hace unity y nosotras solo tenemos el input y el output (la pos del siguiente frame).

In [3]:
#Para ejecutar y guardar nuestros tensores en gpu

print(f"Is the GPU available? {torch.cuda.is_available()}")     
print(f"Is the MPS available? {torch.mps.is_available()}")     # MacOS

if torch.cuda.is_available():
    device = torch.device("cuda") 
elif torch.mps.is_available():
    device = torch.device("mps") 
else:
    device = torch.device("cpu")
print("Device", device)

Is the GPU available? True
Is the MPS available? False
Device cuda


In [4]:
#No basta con crear el device, hay que pushear nuestro tensor al device
x = torch.zeros(2, 3)
x = x.to(device)
print("X", x)

X tensor([[0., 0., 0.],
        [0., 0., 0.]], device='cuda:0')


In [5]:
#Para hacer pruebas entre CPU y GPU

x = torch.randn(5000, 5000)

## CPU version
start_time = time.time()
_ = torch.matmul(x, x)
end_time = time.time()
print(f"CPU time: {(end_time - start_time):6.5f}s")

## GPU / MPS version
x = x.to(device)
_ = torch.matmul(x, x)  # First operation to 'burn in' GPU

# CUDA / MPS is asynchronous, so we need to use different timing functions
if torch.cuda.is_available():
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    _ = torch.matmul(x, x)
    end.record()
    torch.cuda.synchronize()  # Waits for everything to finish running on the GPU
    print(f"GPU time: {0.001 * start.elapsed_time(end):6.5f}s")  # Milliseconds to seconds
elif torch.mps.is_available():
    start = torch.mps.Event(enable_timing=True)
    end = torch.mps.Event(enable_timing=True)
    start.record()
    _ = torch.matmul(x, x)
    end.record()
    torch.mps.synchronize()  # Waits for everything to finish running on the GPU
    print(f"MPS time: {0.001 * start.elapsed_time(end):6.5f}s")  # Milliseconds to seconds    

CPU time: 1.37261s
GPU time: 0.20145s


PACKAGE DE PYTORCH PARA FACILITAR LA CREACIÓN DE NEURAL NETWORKS

Las redes neuronales se componen de modulos, ceamos nuestros propios modulos con las caracteristicas que queramos.

In [6]:
import torch.nn as nn
import torch.nn.functional as F #acceso rapido a funciones
import torch.utils.data as data #cargar y manejar el training data

class MyModule(nn.Module):

    def __init__(self):
        super().__init__()
        # Some init for my module

    def forward(self, x):
        # Function for performing the calculation of the module.
        pass

    #backward se hace automaticamente, podriamos definirla tmbn 

#tmbn clases DataSet y DataLoader

PASOS AL ENTRENAR UN MODELO

1. Get a batch from the data loader
2. Obtain the predictions from the model for the batch
3. Calculate the loss based on the difference between predictions and labels (cn `nn.BCELoss()` o `nn.BCEWithLogitsLoss()`)
4. Backpropagation: calculate the gradients for every parameter with respect to the loss
5. Update the parameters of the model in the direction of the gradients (`torch.optim.SGD`. Stochastic Gradient Descent updates parameters by multiplying the gradients with a small constant, called learning rate, and subtracting those from the parameters (hence minimizing the loss))
6. Save the model so that we can load the same weights at a later time. For this, we extract the so-called `state_dict` from the model which contains all learnable parameters.
7. Evaluate different models to compare.